# DT6 agent: the trees-outside-woodland twin over MCP

DT6 serves the biomass and carbon twin of **trees outside woodland** (lone trees, groups of
trees and small woodlands outside the NFI woodland map) as an MCP server, `dt6_server.py`.
The inventory is the **National Trees Outside Woodland map** (Forest Research 2025); tree
heights come from **Environment Agency lidar** at every epoch flown since 1998 (old LIDAR
Tiles DSMs and the National LIDAR Programme), the only data the twin is calibrated against.
Two growth models, an empirical **size-age curve** indexed by site index and a process-based
**crown light-use model** on **HadUK-Grid** climate, are calibrated crown by crown and project
carbon under baseline, ash-dieback and planting scenarios. Ten 5 km pilot tiles cover the
English regions.

This notebook acts as the orchestrator: it launches the server over stdio, lists its tools and
calls each of them, exactly as an orchestrating agent would.

In [1]:
import sys
from pathlib import Path

from mcp import Client, StdioServerParameters

SERVER = Path("dt6_server.py").resolve()
params = StdioServerParameters(command=sys.executable, args=[str(SERVER)])


async def ask(tool, **args):
    """Launch DT6, call one tool and return its structured result."""
    async with Client(params) as client:
        r = await client.call_tool(tool, args)
        if r.is_error:
            raise RuntimeError(r.content[0].text)
        return r.structured_content

## Discover the tools

In [2]:
async with Client(params) as client:
    for t in (await client.list_tools()).tools:
        print(f"{t.name:22s} {t.description[:90]}")

list_landscapes        List the ten pilot tiles with their landscape, lidar epochs, TOW inventory size, observed 
tree_inventory         The trees outside woodland of a pilot tile or part of it: objects, estimated trees, canopy
carbon_stock           Living biomass carbon (above and below ground) of the trees outside woodland in a pilot ti
canopy_change          Canopy-height change of trees outside woodland between the two lidar epochs nearest the re
attribute_change       Attribute significant canopy-height reductions between two epochs to pressures (felling, t
simulate_trees         Project the trees outside woodland of a tile or area forward with the size-age model, the 
ash_dieback_scenario   Ash dieback scenario for the trees outside woodland of a tile or area: cumulative ash mort
planting_scenario      New tree planting outside woodland (hedgerow trees, agroforestry, riparian buffers or urba
calibrate_area         Crown-by-crown calibration of the growth models in a tile or area

## The ten pilot tiles, their lidar record and calibration skill

In [3]:
ls = await ask("list_landscapes")
print(f"{'pilot':16s} {'region':21s} {'epochs':22s} {'objects':>7s} {'canopy ha':>9s} {'growth m/yr':>11s}  hold-out RMSE size_age / crown_lue / persistence (m)")
for x in ls["landscapes"]:
    c = {k["growth_model"]: k for k in x["calibration"]}
    print(f"{x['key']:16s} {x['region']:21s} {str(x['lidar_epochs']):22s} {x['objects']:7d} {x['canopy_area_ha']:9.1f} "
          f"{x['observed_height_growth_m_yr']:11.2f}  {c['size_age']['holdout_rmse_m']:.2f} / {c['crown_lue']['holdout_rmse_m']:.2f} / {c['size_age']['persistence_rmse_m']:.2f}")

pilot            region                epochs                 objects canopy ha growth m/yr  hold-out RMSE size_age / crown_lue / persistence (m)
breckland        Eastern               [1998, 2009, 2018]        2186      28.3        0.19  2.51 / 2.60 / 3.55
fens             Eastern               [1999, 2017, 2019, 2021]     856      13.4        0.37  2.58 / 2.99 / 2.58
mid_devon        South_West            [2005, 2016, 2020]        6048     161.2        0.07  2.85 / 2.99 / 3.01
somerset_levels  South_West            [2003, 2015, 2021]        6462     102.2        0.15  2.80 / 2.58 / 3.48
herefordshire    West_Midlands         [2002, 2006, 2019]       10190     154.7        0.11  3.88 / 3.87 / 4.34
leicestershire   East_Midlands         [2001, 2014, 2018, 2022]   10561     155.3        0.16  2.23 / 2.16 / 2.07
vale_of_york     Yorkshire_and_Humber  [2000, 2010, 2020]        5766     132.7        0.17  4.22 / 4.15 / 5.75
cheshire         North_West            [2001, 2015, 2021]       18

## State: the inventory and living carbon of a tile (EBCF role: state)

In [4]:
inv = await ask("tree_inventory", pilot="herefordshire")
print(f"Herefordshire tile: {inv['objects']:,} objects, ~{inv['trees']:,} trees, {inv['canopy_area_ha']:.0f} ha canopy "
      f"({inv['canopy_cover_pct']:.1f} % cover), lidar {inv['reference_lidar_year']}")
for r in inv["by_class"]:
    print(f"   {r['tow_class']:15s} {r['objects']:6d} objects  {r['trees']:6d} trees  {r['canopy_area_ha']:6.1f} ha  "
          f"median height {r['height_median_m']:.1f} m  AGB {r['agb_t']:,.0f} t")
for key in ("herefordshire", "oxford", "fens"):
    c = await ask("carbon_stock", pilot=key)
    print(f"{key:14s} living C {c['total_living_c_t']:,.0f} t (AGB 90 % {c['agb_p05_t']:,.0f}-{c['agb_p95_t']:,.0f} t), "
          f"{c['c_t_per_ha_canopy']:.0f} t C per ha canopy, {c['c_t_per_ha_land']:.1f} t C per ha land")
one = await ask("carbon_stock", pilot="cheshire", area={"point": [-2.86, 53.20], "radius_m": 0})
print(f"\nThe single tree object nearest a point in Cheshire: {one['agb_t']:.2f} t AGB, {one['total_living_c_t']:.2f} t C")

Herefordshire tile: 10,190 objects, ~27,704 trees, 155 ha canopy (6.2 % cover), lidar 2019
   lone_tree         7026 objects    7026 trees    28.0 ha  median height 5.9 m  AGB 922 t
   group_of_trees    2799 objects   10899 trees    57.0 ha  median height 8.4 m  AGB 2,136 t
   small_woodland     365 objects    9779 trees    69.8 ha  median height 18.3 m  AGB 6,066 t


herefordshire  living C 5,748 t (AGB 90 % 5,973-12,274 t), 37 t C per ha canopy, 2.3 t C per ha land


oxford         living C 9,810 t (AGB 90 % 10,217-20,927 t), 43 t C per ha canopy, 3.9 t C per ha land


fens           living C 353 t (AGB 90 % 364-756 t), 26 t C per ha canopy, 0.1 t C per ha land



The single tree object nearest a point in Cheshire: 0.81 t AGB, 0.51 t C


## Impact and pressure: canopy change between lidar epochs (EBCF roles: impact, pressure)

In [5]:
ch = await ask("canopy_change", pilot="cheshire", year1=2001, year2=2021)
print(f"Cheshire {ch['epoch1']}-{ch['epoch2']}: {ch['objects_compared']:,} objects compared, median growth "
      f"{ch['median_growth_m_yr']:.2f} m/yr; {ch['significant_reductions']} significant reductions "
      f"({ch['removal_candidates']} removal candidates), {ch['agb_lost_t']:.0f} t AGB lost -> '{ch['impact']}'")
# attribution needs independent evidence; here a made-up felling footprint over the west of the tile
at = await ask("attribute_change", pilot="cheshire", year1=2001, year2=2021,
               evidence=[{"pressure": "felling", "year_from": 2005, "year_to": 2015,
                          "bbox": [-2.90, 53.18, -2.86, 53.23], "source": "illustrative footprint"}])
for a in at["attributions"]:
    print(f"   {a['label']}: {a['attributed_objects']} objects ({a['share_of_reductions']:.0%})")
print(f"   unattributed: {at['unattributed_objects']} ({at['unattributed_share']:.0%})")

Cheshire 2001-2021: 6,765 objects compared, median growth 0.10 m/yr; 198 significant reductions (130 removal candidates), 54 t AGB lost -> 'woody vegetation amount increase'


   woody vegetation amount decrease (felling): 190 objects (96%)
   unattributed: 8 (4%)


## Response: projections with both growth models (EBCF role: response)

In [6]:
r = await ask("simulate_trees", pilot="mid_devon", years=30, n_members=30)
for run in r["runs"]:
    print(f"{run['growth_model']:9s} living C {run['living_c_t']['mean'][0]:,.0f} -> {run['living_c_t']['mean'][-1]:,.0f} t, "
          f"median height {run['height_median_m']['mean'][0]:.1f} -> {run['height_median_m']['mean'][-1]:.1f} m, "
          f"net {run['mean_annual_net_t_co2_yr']:,.0f} t CO2/yr, to soil {run['to_soil_t_c_yr']:.0f} t C/yr")

a = await ask("ash_dieback_scenario", pilot="tyne_valley", growth_model="both", ash_share=0.3,
              replant_share=0.5, years=30, n_members=30)
for base, ash in zip(a["runs"][::2], a["runs"][1::2]):
    print(f"{base['growth_model']:9s} 2055 living C baseline {base['living_c_t']['mean'][-1]:,.0f} t vs ash dieback "
          f"{ash['living_c_t']['mean'][-1]:,.0f} t; cumulative CO2 {base['cumulative_net_t_co2'][0]:,.0f} vs "
          f"{ash['cumulative_net_t_co2'][0]:,.0f} t")

p = await ask("planting_scenario", pilot="somerset_levels", n_trees=1000, kind="riparian", years=50)
for run in p["runs"]:
    print(f"1000 riparian trees, {run['growth_model']:9s}: {run['trees_alive']['mean'][-1]:.0f} alive after 50 yr, "
          f"{run['living_c_t']['mean'][-1]:.0f} t C, cumulative {run['cumulative_net_t_co2'][0]:.0f} t CO2")

size_age  living C 6,492 -> 5,721 t, median height 7.3 -> 8.2 m, net -1,405 t CO2/yr, to soil 399 t C/yr
crown_lue living C 6,647 -> 6,704 t, median height 7.5 -> 9.2 m, net -1,630 t CO2/yr, to soil 432 t C/yr


size_age  2055 living C baseline 5,499 t vs ash dieback 4,494 t; cumulative CO2 -35,206 vs -27,289 t
crown_lue 2055 living C baseline 9,986 t vs ash dieback 8,223 t; cumulative CO2 -62,211 vs -49,764 t


1000 riparian trees, size_age : 679 alive after 50 yr, 13 t C, cumulative -223 t CO2
1000 riparian trees, crown_lue: 679 alive after 50 yr, 20 t C, cumulative -407 t CO2


## Calibration and hold-out validation

In [7]:
c = await ask("calibrate_area", pilot="oxford")
for r in c["results"]:
    print(f"Oxford {r['growth_model']:9s} {r['productivity_name']}: median {r['productivity_median']:.2f} "
          f"(IQR {r['productivity_p25']:.2f}-{r['productivity_p75']:.2f}), tree age median {r['age_median_years']:.0f} yr, "
          f"{r['share_seen_on_2plus_dates']:.0%} seen on 2+ dates, {r['height_drop_events']} height-drop events")
v = await ask("validate")
for r in v["rows"]:
    print(f"{r['pilot']:16s} {r['growth_model']:9s} n {r['n']:5d}  {r['years_ahead_median']:4.1f} yr ahead  RMSE {r['model_rmse_m']:.2f} m "
          f"(persistence {r['persistence_rmse_m']:.2f})  skill {r['skill_vs_persistence']:+.2f}  growth obs {r['observed_growth_m_yr']:.2f} / pred {r['predicted_growth_m_yr']:.2f} m/yr")

Oxford size_age  site index (height at 50 years, m): median 6.38 (IQR 5.38-8.83), tree age median 160 yr, 62% seen on 2+ dates, 10537 height-drop events
Oxford crown_lue light-use efficiency multiplier (0-1): median 0.10 (IQR 0.07-0.13), tree age median 103 yr, 62% seen on 2+ dates, 10537 height-drop events


breckland        size_age  n   544   9.0 yr ahead  RMSE 2.51 m (persistence 3.55)  skill +0.50  growth obs 0.19 / pred 0.23 m/yr
breckland        crown_lue n   544   9.0 yr ahead  RMSE 2.60 m (persistence 3.55)  skill +0.47  growth obs 0.19 / pred 0.37 m/yr
fens             size_age  n   705   1.4 yr ahead  RMSE 2.58 m (persistence 2.58)  skill -0.00  growth obs 0.33 / pred 0.76 m/yr
fens             crown_lue n   705   1.4 yr ahead  RMSE 2.99 m (persistence 2.58)  skill -0.35  growth obs 0.33 / pred 2.03 m/yr
mid_devon        size_age  n  2787  14.3 yr ahead  RMSE 2.85 m (persistence 3.01)  skill +0.10  growth obs 0.07 / pred 0.17 m/yr
mid_devon        crown_lue n  2787  14.3 yr ahead  RMSE 2.99 m (persistence 3.01)  skill +0.02  growth obs 0.07 / pred 0.25 m/yr
somerset_levels  size_age  n  5088   7.0 yr ahead  RMSE 2.80 m (persistence 3.48)  skill +0.35  growth obs 0.26 / pred 0.24 m/yr
somerset_levels  crown_lue n  5088   7.0 yr ahead  RMSE 2.58 m (persistence 3.48)  skill +0.45  g

## Learning from new heights with the Kalman filter

In [8]:
import numpy as np
sys.path.insert(0, ".")
from trees import inventory
inv_b = inventory.load("breckland")
obs = [{"tow_id": str(inv_b.tow_id[i]), "year": 2026.0, "height_m": float(inv_b.h_fz[i]) + 1.5, "sd_m": 0.5}
       for i in range(40)]   # an illustrative survey that finds the trees 1.5 m taller
a = await ask("assimilate_heights", pilot="breckland", site="demo-farm", observations=obs, persist=False)
for r in a["results"]:
    print(f"{r['growth_model']:9s} matched {r['n_matched']} objects, productivity shift {r['productivity_shift']:+.3f} "
          f"+- {r['productivity_shift_sd']:.3f} ({r['productivity_name']})")
g = await ask("get_parameters", pilot="breckland")
for p in g["parameters"]:
    print(f"{p['growth_model']:9s} {p['productivity_median']:.3f} +- {p['productivity_sd']:.3f}: {p['source']}")

size_age  matched 40 objects, productivity shift +0.194 +- 0.050 (site index (height at 50 years, m))
crown_lue matched 40 objects, productivity shift +0.014 +- 0.002 (light-use efficiency multiplier (0-1))


size_age  8.780 +- 1.329: calibrated baseline: crown-by-crown EA lidar calibration in breckland
crown_lue 0.191 +- 0.016: calibrated baseline: crown-by-crown EA lidar calibration in breckland
